# 14 — Streaming Pipeline (Amazon FAR style)

A multi-stage processing pipeline (think ETL / stream processing). Parts 1-3 build the core
(concurrency at Part 3); Parts 4-5 are stretch tasks (error routing, then a parallel CPU-bound stage).
Fill stubs, run each test cell, peek at the collapsed `ref_` solutions only after trying.

A "stage" is just a function `item -> item`; a pipeline is a list of stages applied in order.

---

## Part 1 — One stage

`run_stage(fn, items) -> list`: apply `fn` to every item, returning the transformed list (order
preserved).

**Lock down:** a stage is a pure transform; this is the unit the pipeline composes.

In [ ]:
def run_stage(fn, items):
    raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    assert run_stage(lambda x: x * 2, [1, 2, 3]) == [2, 4, 6]
    assert run_stage(str, [1, 2]) == ["1", "2"]
    assert run_stage(lambda x: x, []) == []
    print("Part 1 OK")

_t1()

## Part 2 — Compose stages

`run_pipeline(stages, items) -> list`: feed items through each stage in order (output of one is input
to the next).

**Lock down:** an empty pipeline is the identity; composition order matters; still single-threaded and
order-preserving.

In [ ]:
def run_pipeline(stages, items):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    stages = [lambda x: x + 1, lambda x: x * 2, lambda x: -x]
    assert run_pipeline(stages, [1, 2, 3]) == [-4, -6, -8]
    assert run_pipeline([], [1, 2]) == [1, 2]
    print("Part 2 OK")

_t2()

## Part 3 — Concurrent pipeline

`run_pipeline_concurrent(stages, items, queue_size=8) -> list`: run **each stage in its own thread**,
connected by **bounded** `queue.Queue`s, so stages process different items at the same time. A sentinel
flows through to shut each stage down in turn. Output order must match input order, and the result must
equal `run_pipeline`.

**Discuss:** one worker per stage + FIFO queues preserves order; bounded queues give **backpressure**
(a slow stage throttles upstream); sentinel propagation for clean shutdown; pipeline parallelism vs
data parallelism.

In [ ]:
import queue
import threading


def run_pipeline_concurrent(stages, items, queue_size=8):
    raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    stages = [lambda x: x + 1, lambda x: x * 2]
    assert run_pipeline_concurrent(stages, list(range(20))) == [(i + 1) * 2 for i in range(20)]
    assert run_pipeline_concurrent([], [1, 2, 3]) == [1, 2, 3]
    assert run_pipeline_concurrent([lambda x: x], list(range(100))) == list(range(100))  # order kept
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Error routing

`run_pipeline_safe(stages, items) -> (outputs, errors)`: push each item through all stages, but if a
stage raises, **route that item to an error sink** (record `(original_item, exception)`) and drop it
from the pipeline — one bad item must not kill the run. `outputs` are the items that made it all the
way through.

**Discuss:** dead-letter queues, where to attach the error sink, and per-item vs batch failure
isolation.

In [ ]:
def run_pipeline_safe(stages, items):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    def stage2(x):
        if x == 3:
            raise ValueError("bad 3")
        return x * 10

    stages = [lambda x: x + 1, stage2]
    out, errs = run_pipeline_safe(stages, [1, 2, 2])   # 1->2->20 ok; 2->3->raise (x2)
    assert out == [20]
    assert [orig for orig, _ in errs] == [2, 2]        # original (pre-stage) items recorded
    assert all(isinstance(e, ValueError) for _, e in errs)
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel CPU-bound stage

`parallel_stage(items, num_procs) -> list`: run a CPU-bound stage over all items across processes with
`ProcessPoolExecutor`, preserving order; worker is `pipeline_workers.heavy_stage`.

**Discuss:** in a real pipeline the I/O-bound stages stay on threads/async while a CPU-bound stage
fans out to processes (a hybrid); the GIL is why; pickling cost at the process boundary.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import pipeline_workers


def parallel_stage(items, num_procs) -> list:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    items = [1, 2, 3, 10]
    assert parallel_stage(items, 2) == [sum(i * i for i in range(n)) for n in items]
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Multiple workers per stage (data parallelism) and how it breaks ordering — restore with sequence tags.
- Batching between stages; windowing; flush-on-shutdown.
- async pipelines (`asyncio.Queue`); fan-out / fan-in topologies.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import queue
import threading
from concurrent.futures import ProcessPoolExecutor
import pipeline_workers


def ref_run_stage(fn, items):
    return [fn(x) for x in items]


def ref_run_pipeline(stages, items):
    data = list(items)
    for fn in stages:
        data = [fn(x) for x in data]
    return data


def ref_run_pipeline_concurrent(stages, items, queue_size=8):
    sentinel = object()
    queues = [queue.Queue(maxsize=queue_size) for _ in range(len(stages) + 1)]

    def feeder():
        for x in items:
            queues[0].put(x)
        queues[0].put(sentinel)

    def stage_worker(i, fn):
        q_in, q_out = queues[i], queues[i + 1]
        while True:
            x = q_in.get()
            if x is sentinel:
                q_out.put(sentinel)                   # propagate shutdown downstream
                return
            q_out.put(fn(x))

    threads = [threading.Thread(target=feeder)]
    for i, fn in enumerate(stages):
        threads.append(threading.Thread(target=stage_worker, args=(i, fn)))
    for t in threads:
        t.start()

    out = []
    while True:
        x = queues[-1].get()
        if x is sentinel:
            break
        out.append(x)
    for t in threads:
        t.join()
    return out


def ref_run_pipeline_safe(stages, items):
    outputs, errors = [], []
    for x in items:
        cur, ok = x, True
        for fn in stages:
            try:
                cur = fn(cur)
            except Exception as e:                    # noqa: BLE001 — route to the error sink
                errors.append((x, e))
                ok = False
                break
        if ok:
            outputs.append(cur)
    return outputs, errors


def ref_parallel_stage(items, num_procs):
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        return list(ex.map(pipeline_workers.heavy_stage, items))


assert ref_run_stage(lambda x: x + 1, [0, 9]) == [1, 10]
assert ref_run_pipeline([lambda x: x * 2, lambda x: x + 1], [1, 2]) == [3, 5]
assert ref_run_pipeline_concurrent([lambda x: x * 2], list(range(10))) == [2 * i for i in range(10)]
_o, _e = ref_run_pipeline_safe([lambda x: 1 // x], [1, 0, 2])
assert _o == [1, 0] and len(_e) == 1 and _e[0][0] == 0
assert ref_parallel_stage([1, 5], 2) == [0, 30]
print("reference solutions OK")